# 📊 基于矩阵不可约性（Matrix Irreducibility）的供应链与经销商网络卡顿点诊断

### 💡 项目背景与分析目标

在食品与消费品等制造型企业中，工厂投向渠道的资源（引流品、利润品、返利补贴与营销费用）能否在经销商网络中形成“自循环闭环”，决定了整个供应链的健康度与 ROI。

传统进销存分析仅能看到静态的库存与销量，本 Notebook 采用**图论（Graph Theory）**与**马尔可夫转移矩阵（Markov Transition Matrix）**，将货品与资金的流转抽象为有向图拓扑网络，利用**矩阵不可约性（Irreducibility）**对全网生态进行动态“全景 CT 扫描”：

1. **结构闭环诊断（不可约性）**：验证货物流转网络是否构成强连通图（Strongly Connected Graph）。若矩阵不可约，说明系统具备自造血闭环能力；若矩阵可约，则意味着存在渠道割裂。
2. **瓶颈与黑洞定位（吸收态分析）**：自动精准定位只进不出、囤货蹿货或拒绝卖利润品的“吸收态节点（Absorbing Nodes）”与死胡同。
3. **滞留时间定量测量（基本矩阵 $N$）**：运用马尔可夫链基本矩阵 $N = (I - Q)^{-1}$，精确计算货物在被黑洞节点吸收前，在各个经销商与产品线之间的**平均卡顿滞留周期**。

---


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# 设置随机种子以确保演示可复现
np.random.seed(42)

# ==============================================================================
# 1. 模拟企业 ERP / DMS 原始交易日志数据 (Raw Transaction Event Logs)
# ==============================================================================


def generate_mock_erp_logs(num_orders=1000, is_healthy_ecosystem=True):
    """
    模拟生成经销商的履约与复购事件流
    节点定义：
    - Node 0: 引流品 (Traffic Product - 低毛利/高频通货)
    - Node 1: 利润品 (Profit Product - 高毛利/核心商品)
    - Node 2: 渠道返利与补贴 (Rebate & Marketing Subsidy)
    - Node 3: 渠道沉淀/新品研发 (Channel Retention / R&D)
    """
    distributor_ids = [f"DIST_{i:03d}" for i in range(1, 21)]  # 20家经销商
    node_names = ["0_引流品", "1_利润品", "2_返利补贴", "3_渠道沉淀"]

    # 模拟健康与病态状态下的真实转化的先验概率分布
    if is_healthy_ecosystem:
        # 健康状态：完整的闭环生态 (不可约)
        true_transition_probs = {
            "0_引流品":  [0.2, 0.6, 0.2, 0.0],
            "1_利润品":  [0.0, 0.2, 0.7, 0.1],
            "2_返利补贴": [0.1, 0.0, 0.3, 0.6],
            "3_渠道沉淀": [0.7, 0.3, 0.0, 0.0]  # 沉淀能够反哺引流品与利润品
        }
    else:
        # 病态状态：节点3（渠道沉淀）变成了吸收态/黑洞，不反哺引流品 (可约)
        true_transition_probs = {
            "0_引流品":  [0.2, 0.8, 0.0, 0.0],
            "1_利润品":  [0.0, 0.3, 0.7, 0.0],
            "2_返利补贴": [0.0, 0.0, 0.4, 0.6],
            "3_渠道沉淀": [0.0, 0.0, 0.0, 1.0]  # 只进不出！
        }

    logs = []
    for dist in distributor_ids:
        # 每个经销商从随机节点开始
        current_state = np.random.choice(node_names)
        for step in range(num_orders // len(distributor_ids)):
            next_state = np.random.choice(
                node_names,
                p=true_transition_probs[current_state]
            )
            logs.append({
                "distributor_id": dist,
                "current_state": current_state,
                "next_state": next_state,
                "amount": np.random.randint(1000, 50000)
            })
            current_state = next_state

    return pd.DataFrame(logs), node_names


# ==============================================================================
# 2. 核心数学管道：从数据日志构建转移概率矩阵
# ==============================================================================
def build_transition_matrix_from_logs(df_logs, node_names):
    """
    从原始事件日志提取并计算行归一化的转移概率矩阵 P
    """
    n = len(node_names)
    # 初始化频次计数矩阵
    counts = pd.crosstab(
        df_logs['current_state'],
        df_logs['next_state']
    ).reindex(index=node_names, columns=node_names, fill_value=0)

    # 行归一化 (Row-Normalization) 得到概率矩阵 P
    row_sums = counts.sum(axis=1)
    # 防止除以 0
    row_sums[row_sums == 0] = 1.0
    P = counts.div(row_sums, axis=0).to_numpy()

    return P, counts


# ==============================================================================
# 3. 极核引擎：基于图论与矩阵分析的渠道健康度全景诊断
# ==============================================================================
class ChannelEcosystemAnalyzer:
    def __init__(self, P, node_names):
        self.P = np.array(P)
        self.node_names = node_names
        self.n = len(node_names)
        self.G = nx.DiGraph()

        # 构建加权有向图
        for i in range(self.n):
            for j in range(self.n):
                if self.P[i, j] > 0:
                    self.G.add_edge(
                        self.node_names[i], self.node_names[j], weight=self.P[i, j])

    def check_irreducibility(self):
        """1. 检查矩阵不可约性 (利用强连通性)"""
        return nx.is_strongly_connected(self.G)

    def find_absorbing_and_dead_ends(self):
        """2. 定位系统中的吸收态节点与死胡同"""
        absorbing_nodes = []
        for i, name in enumerate(self.node_names):
            # 自环概率接近 1 且对其他节点无输出
            if np.isclose(self.P[i, i], 1.0) or (np.sum(self.P[i, :]) - self.P[i, i] == 0):
                absorbing_nodes.append(name)
        return absorbing_nodes

    def compute_stationary_distribution(self):
        """3. 求解长期稳态资源分布向量 (主特征向量)"""
        if not self.check_irreducibility():
            return None

        # 求解 P^T v = v，寻找λ=1对应的左特征向量
        eigenvalues, eigenvectors = np.linalg.eig(self.P.T)
        idx = np.argmin(np.abs(eigenvalues - 1.0))
        stationary = np.real(eigenvectors[:, idx])
        stationary = stationary / np.sum(stationary)  # 归一化
        return stationary

    def compute_fundamental_matrix(self, transient_indices, absorbing_indices):
        """
        4. 计算马尔可夫链基本矩阵 N = (I - Q)^(-1)
        用于精准测量资源在卡顿节点/暂态节点的平均滞留时间与流失概率
        """
        Q = self.P[np.ix_(transient_indices, transient_indices)]
        I = np.eye(len(transient_indices))
        N = np.linalg.inv(I - Q)
        return N

    def generate_full_report(self):
        """5. 输出结构化的 CEO / 战略层决策报告"""
        is_irreducible = self.check_irreducibility()
        absorbing_nodes = self.find_absorbing_and_dead_ends()
        sccs = list(nx.strongly_connected_components(self.G))

        print("\n" + "=" * 70)
        print("          🏥 渠道生态矩阵与供应链健康度诊断报告")
        print("=" * 70)

        # 诊断一：不可约性诊断
        print("\n【维度一：生态拓扑结构与不可约性】")
        if is_irreducible:
            print("  🟢 结论：矩阵为【不可约矩阵 (Irreducible)】")
            print("  🟢 状态：整个渠道网络构成了强连通闭环生态！资源/流量在各节点间无阻碍循环。")
        else:
            print("  🔴 结论：矩阵为【可约矩阵 (Reducible)】")
            print("  🔴 状态：渠道网络已断流或割裂！存在单向耗损或资源沉淀死角。")
            print(f"  ⚠️  强连通子集拆解 ({len(sccs)} 个孤岛): {sccs}")

        # 诊断二：黑洞节点定位
        print("\n【维度二：断流黑洞与吸收态节点定位】")
        if absorbing_nodes:
            for node in absorbing_nodes:
                print(f"  🛑 [致命警告] 检测到黑洞节点: '{node}' ！资源进入后回收率为 0%。")
        else:
            print("  ✅ 全网未检测到彻底死锁的吸收态节点。")

        # 诊断三：稳态分布与效率预测
        print("\n【维度三：长期稳态资源分布与循环效率】")
        if is_irreducible:
            pi = self.compute_stationary_distribution()
            df_pi = pd.DataFrame({
                "节点名称": self.node_names,
                "稳态资源占比": [f"{p*100:.2f}%" for p in pi],
                "平均回流周期 (Step)": [f"{1/p:.2f} 步" for p in pi]
            })
            print(df_pi.to_string(index=False))
        else:
            print("  ⚠️ 矩阵可约，全局稳态无法收敛（资源将全部沉淀至吸收态黑洞中）。")

            # 定量计算卡在暂态节点的平均时间
            transient_idx = [i for i in range(
                self.n) if self.node_names[i] not in absorbing_nodes]
            absorbing_idx = [i for i in range(
                self.n) if self.node_names[i] in absorbing_nodes]

            if transient_idx and absorbing_idx:
                N = self.compute_fundamental_matrix(
                    transient_idx, absorbing_idx)
                avg_steps = N.sum(axis=1)
                print("\n  📉 [暂态瓶颈分析] 流量在被黑洞彻底吞噬前，在各节点的平均滞留步数:")
                for idx, steps in zip(transient_idx, avg_steps):
                    print(
                        f"     * 从 '{self.node_names[idx]}' 出发，卡顿滞留时间: {steps:.2f} 次交易周期")


# ==============================================================================
# 4. 主流程执行与双场景对比实测
# ==============================================================================
if __name__ == "__main__":
    print(">>> 正在初始化企业 ERP 渠道交易日志引擎...")

    # --------------------------------------------------------------------------
    # 场景 A：健康闭环渠道生态实测
    # --------------------------------------------------------------------------
    df_healthy_logs, nodes = generate_mock_erp_logs(
        num_orders=2000, is_healthy_ecosystem=True)
    P_healthy, _ = build_transition_matrix_from_logs(df_healthy_logs, nodes)

    print("\n" + "▶" * 35 + " 测试场景 A：健康闭环生态 " + "◀" * 35)
    analyzer_A = ChannelEcosystemAnalyzer(P_healthy, nodes)
    analyzer_A.generate_full_report()

    # --------------------------------------------------------------------------
    # 场景 B：畸形/断流渠道生态实测 (渠道沉淀变成吸收态黑洞)
    # --------------------------------------------------------------------------
    df_sick_logs, nodes = generate_mock_erp_logs(
        num_orders=2000, is_healthy_ecosystem=False)
    P_sick, _ = build_transition_matrix_from_logs(df_sick_logs, nodes)

    print("\n\n" + "▶" * 35 + " 测试场景 B：畸形断流生态 " + "◀" * 35)
    analyzer_B = ChannelEcosystemAnalyzer(P_sick, nodes)
    analyzer_B.generate_full_report()

>>> 正在初始化企业 ERP 渠道交易日志引擎...

▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶ 测试场景 A：健康闭环生态 ◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀

          🏥 渠道生态矩阵与供应链健康度诊断报告

【维度一：生态拓扑结构与不可约性】
  🟢 结论：矩阵为【不可约矩阵 (Irreducible)】
  🟢 状态：整个渠道网络构成了强连通闭环生态！资源/流量在各节点间无阻碍循环。

【维度二：断流黑洞与吸收态节点定位】
  ✅ 全网未检测到彻底死锁的吸收态节点。

【维度三：长期稳态资源分布与循环效率】
  节点名称 稳态资源占比 平均回流周期 (Step)
 0_引流品 23.17%        4.32 步
 1_利润品 24.44%        4.09 步
2_返利补贴 31.40%        3.18 步
3_渠道沉淀 20.98%        4.77 步


▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶ 测试场景 B：畸形断流生态 ◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀◀

          🏥 渠道生态矩阵与供应链健康度诊断报告

【维度一：生态拓扑结构与不可约性】
  🔴 结论：矩阵为【可约矩阵 (Reducible)】
  🔴 状态：渠道网络已断流或割裂！存在单向耗损或资源沉淀死角。
  ⚠️  强连通子集拆解 (4 个孤岛): [{'3_渠道沉淀'}, {'2_返利补贴'}, {'1_利润品'}, {'0_引流品'}]

【维度二：断流黑洞与吸收态节点定位】
  🛑 [致命警告] 检测到黑洞节点: '3_渠道沉淀' ！资源进入后回收率为 0%。

【维度三：长期稳态资源分布与循环效率】
  ⚠️ 矩阵可约，全局稳态无法收敛（资源将全部沉淀至吸收态黑洞中）。

  📉 [暂态瓶颈分析] 流量在被黑洞彻底吞噬前，在各节点的平均滞留步数:
     * 从 '0_引流品' 出发，卡顿滞留时间: 4.40 次交易周期
     * 从 '1_利润品' 出发，卡顿滞留时间: 3.03 次交易周期
     * 从 '2_返利补贴' 出发，卡顿滞留时间: 1.58 次交易周期
